# Bloco 03 — Agregações e Joins

Neste bloco vamos dominar duas operações fundamentais do PySpark:

1. **Agregações** — `groupBy()`, `.agg()`, `orderBy()`, `limit()`
2. **Joins** — inner, left, right, full, anti, semi e broadcast

Você já sabe fazer tudo isso em SQL. Agora vai aprender a sintaxe equivalente em PySpark DataFrame API, aplicada ao dataset Northwind.

---

## Setup

In [0]:
# pyspark.sql.functions — modulo principal de funcoes do PySpark.
# Contem funcoes para manipulacao de colunas, agregacoes, strings, datas, etc.
# Importamos como "F" por convencao para evitar conflito com built-ins do Python.
from pyspark.sql import functions as F

# broadcast() — funcao de otimizacao de joins.
# Quando uma tabela e pequena (ex: categories, shippers), o broadcast()
# replica ela em todos os nos do cluster, evitando o shuffle (operacao cara).
# Importamos diretamente porque e usada como wrapper: broadcast(df_pequeno).
from pyspark.sql.functions import broadcast

CATALOG = "northwind"
SCHEMA = "bronze"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Carregando todas as tabelas do Northwind como DataFrames
orders = spark.table("orders")
order_details = spark.table("order_details")
customers = spark.table("customers")
products = spark.table("products")
categories = spark.table("categories")
employees = spark.table("employees")
shippers = spark.table("shippers")
suppliers = spark.table("suppliers")

print("Tabelas carregadas com sucesso!")

---

## 1. GroupBy + Agregações

Em SQL você usa `GROUP BY ... COUNT(*), SUM(col), AVG(col)`.  
Em PySpark, a lógica é a mesma: `df.groupBy("col").agg(...)`.

### Agregações simples

O PySpark permite chamar agregações diretamente após `groupBy()`:

In [0]:
# count() — contagem de pedidos por employee_id
orders.groupBy("employee_id").count().show()

In [0]:
# sum() — soma do frete por país de destino
orders.groupBy("ship_country").sum("freight").show(5)

In [0]:
# avg() — média do frete por país de destino
orders.groupBy("ship_country").avg("freight").show(5)

In [0]:
# min() e max() — menor e maior frete por país
orders.groupBy("ship_country").min("freight").show(5)
orders.groupBy("ship_country").max("freight").show(5)

### Múltiplas agregações com `.agg()`

Para aplicar várias agregações de uma vez, use `.agg()` com `.alias()` para nomear as colunas.

**Equivalente SQL:**
```sql
SELECT employee_id,
       COUNT(*) AS total_pedidos,
       ROUND(AVG(freight), 2) AS frete_medio,
       ROUND(SUM(freight), 2) AS frete_total
FROM orders
GROUP BY employee_id
```

In [0]:
# Múltiplas agregações: contagem, média e soma de frete por funcionário
pedidos_por_employee = (
    orders
    .groupBy("employee_id")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(F.avg("freight"), 2).alias("frete_medio"),
        F.round(F.sum("freight"), 2).alias("frete_total")
    )
)

pedidos_por_employee.show()

In [0]:
# Média do frete por país de destino
frete_por_pais = (
    orders
    .groupBy("ship_country")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(F.avg("freight"), 2).alias("frete_medio")
    )
)

frete_por_pais.show(10)

### `approx_count_distinct` — Contagem aproximada

Para datasets grandes, `countDistinct` e caro (precisa de shuffle completo). O `approx_count_distinct` usa o algoritmo **HyperLogLog** para uma estimativa com ~2-3% de margem de erro, mas muito mais rapido.

In [0]:
# Comparar countDistinct vs approx_count_distinct
orders.agg(
    F.countDistinct("customer_id").alias("count_distinct_exato"),
    F.approx_count_distinct("customer_id").alias("approx_count_distinct"),
    F.count("*").alias("total_pedidos")
).show()

# approx_count_distinct por país
orders.groupBy("ship_country").agg(
    F.approx_count_distinct("customer_id").alias("clientes_aprox"),
    F.count("*").alias("total_pedidos")
).orderBy(F.col("total_pedidos").desc()).show(10)

---

## 2. OrderBy / Sort

Equivalente ao `ORDER BY` do SQL. Funciona com `.desc()` e `.asc()`, e aceita múltiplas colunas.

In [0]:
# Ordenar frete médio por país — descendente
frete_por_pais.orderBy(F.col("frete_medio").desc()).show(10)

In [0]:
# Ordenar por múltiplas colunas: primeiro total_pedidos DESC, depois frete_medio ASC
(
    frete_por_pais
    .orderBy(
        F.col("total_pedidos").desc(),
        F.col("frete_medio").asc()
    )
    .show(10)
)

In [0]:
# limit(n) — equivalente ao TOP / LIMIT do SQL
# Top 5 países com maior frete médio
(
    frete_por_pais
    .orderBy(F.col("frete_medio").desc())
    .limit(5)
    .show()
)

---

## 3. Relatório 1 — Receita Total de 1997

Vamos reconstruir o equivalente PySpark da view `total_revenues_1997_view`.

**Passos:**
1. Calcular a receita por item em `order_details`
2. Filtrar pedidos de 1997
3. Fazer join e somar

**Fórmula de receita:**  
`receita = unit_price * quantity * (1.0 - discount)`

In [0]:
# Passo 1: Criar coluna de receita em order_details
od_receita = order_details.withColumn(
    "receita",
    F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
)

# Passo 2: Filtrar pedidos de 1997
orders_1997 = orders.filter(F.year(F.col("order_date")) == 1997)

# Passo 3: Join order_details (com receita) + orders de 1997, e somar
receita_1997 = (
    od_receita
    .join(orders_1997, "order_id")
    .agg(F.round(F.sum("receita"), 2).alias("receita_total_1997"))
)

receita_1997.show()

### Equivalente SQL

```sql
SELECT ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS receita_total_1997
FROM order_details od
INNER JOIN orders o ON od.order_id = o.order_id
WHERE YEAR(o.order_date) = 1997;
```

---

## 4. Tipos de Join

O PySpark suporta **7 tipos de join**. Você já conhece `INNER`, `LEFT`, `RIGHT` e `FULL` do SQL. Agora vamos ver todos, incluindo `anti` e `semi`, que são muito úteis.

| Tipo | Descrição |
|------|----------|
| `inner` | Apenas registros que existem nas duas tabelas |
| `left` | Todos da esquerda + match da direita |
| `right` | Todos da direita + match da esquerda |
| `full` / `outer` | Todos de ambas as tabelas |
| `anti` | Registros da esquerda que **NÃO** têm match na direita |
| `semi` | Registros da esquerda que **TÊM** match, sem duplicar colunas |

### 4.1 Inner Join

Retorna apenas os registros que existem em ambas as tabelas.  
**Northwind:** Pedidos com seus respectivos clientes.

In [0]:
# Inner Join: pedidos com dados do cliente
pedidos_com_clientes = (
    orders
    .join(customers, "customer_id", "inner")
    .select(
        "order_id",
        "customer_id",
        "company_name",
        "order_date",
        "ship_country"
    )
)

print(f"Total de pedidos com cliente: {pedidos_com_clientes.count()}")
pedidos_com_clientes.show(5)

### 4.2 Left Join

Retorna **todos** os registros da tabela da esquerda, mesmo sem match na direita.  
**Northwind:** Todos os clientes, mesmo os que nunca fizeram pedido.

In [0]:
# Left Join: todos os clientes, com ou sem pedidos
clientes_com_pedidos = (
    customers
    .join(orders, "customer_id", "left")
    .select(
        "customer_id",
        "company_name",
        "order_id",
        "order_date"
    )
)

# Clientes sem pedidos terão order_id = null
sem_pedidos = clientes_com_pedidos.filter(F.col("order_id").isNull())
print(f"Clientes sem nenhum pedido: {sem_pedidos.count()}")
sem_pedidos.show(5)

### 4.3 Right Join

Retorna **todos** os registros da tabela da direita, mesmo sem match na esquerda.  
É o inverso do left join. Na prática, é mais comum usar `left` e trocar a ordem das tabelas.

In [0]:
# Right Join: todos os pedidos, com ou sem cliente cadastrado
pedidos_right = (
    customers
    .join(orders, "customer_id", "right")
    .select(
        "order_id",
        "customer_id",
        "company_name",
        "order_date"
    )
)

print(f"Total de registros (right join): {pedidos_right.count()}")
pedidos_right.show(5)

### 4.4 Full / Outer Join

Retorna **todos** os registros de ambas as tabelas, com ou sem match.

In [0]:
# Full Outer Join: combinação completa
full_join = (
    customers
    .join(orders, "customer_id", "full")
    .select(
        "customer_id",
        "company_name",
        "order_id",
        "order_date"
    )
)

print(f"Total de registros (full join): {full_join.count()}")
full_join.show(5)

### 4.5 Anti Join

Retorna registros da tabela da esquerda que **NÃO** têm correspondência na direita.  
**Northwind:** Clientes que NUNCA fizeram pedido.

**Equivalente SQL:**
```sql
SELECT c.*
FROM customers c
WHERE c.customer_id NOT IN (SELECT DISTINCT customer_id FROM orders)
```

In [0]:
# Anti Join: clientes que NUNCA fizeram pedido
clientes_sem_pedidos = customers.join(
    orders,
    customers.customer_id == orders.customer_id,
    "anti"
)

print(f"Clientes que nunca compraram: {clientes_sem_pedidos.count()}")
clientes_sem_pedidos.select("customer_id", "company_name", "country").show()

### 4.6 Semi Join

Retorna registros da tabela da esquerda que **TÊM** correspondência na direita, mas **sem duplicar colunas** da direita.  
**Northwind:** Clientes que fizeram pelo menos 1 pedido (sem trazer colunas de orders).

**Equivalente SQL:**
```sql
SELECT c.*
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
```

In [0]:
# Semi Join: clientes que fizeram pelo menos 1 pedido
# Diferente do inner join: não duplica linhas e não traz colunas de orders
clientes_com_pedidos_semi = customers.join(
    orders,
    customers.customer_id == orders.customer_id,
    "semi"
)

print(f"Clientes que já compraram (semi join): {clientes_com_pedidos_semi.count()}")
print(f"Colunas retornadas: {clientes_com_pedidos_semi.columns}")
clientes_com_pedidos_semi.select("customer_id", "company_name", "country").show(5)

---

## 5. Broadcast Join

Quando uma das tabelas é **pequena** (tabelas de dimensão como `categories`, `shippers`), o Spark pode replicar essa tabela em todos os nós do cluster. Isso evita o **shuffle** (redistribuição de dados entre nós), que é a operação mais cara em joins.

Use `broadcast()` como hint para forçar esse comportamento:

```python
from pyspark.sql.functions import broadcast
df_grande.join(broadcast(df_pequeno), "chave")
```

**Quando usar:** Tabelas de referência/dimensão (categories, shippers, etc.)  
**Quando NÃO usar:** Tabelas grandes (orders, order_details)

---

## 5.1 Union — Combinar DataFrames verticalmente

`union()` empilha dois DataFrames com o **mesmo schema** (mesmo numero de colunas, mesmos tipos).

- `union()` — mantém duplicatas (equivalente a `UNION ALL` do SQL)
- `unionByName()` — alinha por nome de coluna (mais seguro quando a ordem muda)

> **Nota:** `union()` no PySpark ja funciona como `UNION ALL` do SQL (NAO remove duplicatas automaticamente).

In [0]:
# Union: combinar pedidos de 1996 e 1997 separadamente
orders_1996 = orders.filter(F.year("order_date") == 1996).select("order_id", "customer_id", "order_date")
orders_1997 = orders.filter(F.year("order_date") == 1997).select("order_id", "customer_id", "order_date")

print(f"Pedidos 1996: {orders_1996.count()}")
print(f"Pedidos 1997: {orders_1997.count()}")

# union() — empilha os dois DataFrames
combined = orders_1996.union(orders_1997)
print(f"Union (1996 + 1997): {combined.count()}")

# Para remover duplicatas após union, use .distinct()
combined_sem_duplicatas = combined.distinct()
print(f"Union + distinct: {combined_sem_duplicatas.count()}")

In [0]:
# Broadcast Join: categories é pequena, então replicamos em todos os nós
produtos_com_categoria = (
    products
    .join(broadcast(categories), "category_id")
    .select(
        "product_id",
        "product_name",
        "category_name",
        "unit_price"
    )
)

produtos_com_categoria.show(10)

In [0]:
# Verificando o plano de execução — observe o BroadcastHashJoin
produtos_com_categoria.explain()

---

## 6. Relatório 2 — Receita por Cliente

Equivalente PySpark da view `view_total_revenues_per_customer`.  
Join de 3 tabelas: `order_details` + `orders` + `customers`, agrupado por `company_name`.

In [0]:
# Receita total por cliente
receita_por_cliente = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .groupBy("company_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
    .orderBy(F.col("total").desc())
)

print(f"Total de clientes: {receita_por_cliente.count()}")
receita_por_cliente.show(10)

### Equivalente SQL

```sql
SELECT c.company_name,
       ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS total
FROM order_details od
INNER JOIN orders o ON od.order_id = o.order_id
INNER JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.company_name
ORDER BY total DESC;
```

---

## 7. Relatório 3 — Top 10 Produtos

Equivalente PySpark da view `top_10_products`.  
Join `order_details` + `products`, agrupado por `product_name`, limitado a 10.

In [0]:
# Top 10 produtos por receita
top_10 = (
    od_receita
    .join(products, "product_id")
    .groupBy("product_name")
    .agg(F.round(F.sum("receita"), 2).alias("sales"))
    .orderBy(F.col("sales").desc())
    .limit(10)
)

top_10.show()

### Equivalente SQL

```sql
SELECT p.product_name,
       ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS sales
FROM order_details od
INNER JOIN products p ON od.product_id = p.product_id
GROUP BY p.product_name
ORDER BY sales DESC
LIMIT 10;
```

---

## 8. Filtro pós-agregação (HAVING)

Em SQL, para filtrar **depois** de agregar, você usa `HAVING`.  
Em PySpark, não existe `HAVING` como keyword. A solução é simples: basta usar outro `.filter()` **depois** do `.agg()`.

```python
# SQL:     GROUP BY col HAVING SUM(x) > 1000
# PySpark: .groupBy("col").agg(F.sum("x").alias("total")).filter(F.col("total") > 1000)
```

Isso funciona porque em PySpark cada transformação retorna um novo DataFrame, então você pode encadear quantos `.filter()` quiser.

### Relatório 4 — Clientes UK com pagamento > 1000

Equivalente PySpark da view `uk_clients_who_pay_more_then_1000`.  

**Lógica:**
1. Filtrar clientes do UK (`.filter()` pré-agregação = `WHERE`)
2. Agrupar por `contact_name` e somar receita
3. Filtrar soma > 1000 (`.filter()` pós-agregação = `HAVING`)

In [0]:
# Clientes UK com pagamento total > 1000
uk_clientes = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .filter(F.lower(F.col("country")) == "uk")           # WHERE country = 'uk'
    .groupBy("contact_name")
    .agg(F.round(F.sum("receita"), 2).alias("payments"))
    .filter(F.col("payments") > 1000)                     # HAVING payments > 1000
    .orderBy(F.col("payments").desc())
)

uk_clientes.show()

### Equivalente SQL

```sql
SELECT c.contact_name,
       ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS payments
FROM order_details od
INNER JOIN orders o ON od.order_id = o.order_id
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE LOWER(c.country) = 'uk'
GROUP BY c.contact_name
HAVING payments > 1000
ORDER BY payments DESC;
```

**Observe:** Em PySpark, o `HAVING` é apenas outro `.filter()` após `.agg()`. Não existe keyword separada.

---

## 9. Resumo dos Joins

| Join | Sintaxe PySpark | Quando usar |
|------|----------------|-------------|
| Inner | `df1.join(df2, "key")` | Apenas registros com match |
| Left | `df1.join(df2, "key", "left")` | Todos da esquerda + match |
| Right | `df1.join(df2, "key", "right")` | Todos da direita + match |
| Full | `df1.join(df2, "key", "full")` | Tudo de ambas |
| Anti | `df1.join(df2, cond, "anti")` | Quem NÃO tem match |
| Semi | `df1.join(df2, cond, "semi")` | Quem TEM match, sem duplicar colunas |
| Broadcast | `df1.join(broadcast(df2), "key")` | Tabela pequena + grande |

**Dica:** No anti e semi join, use a sintaxe explícita da condição (`df1.col == df2.col`) para evitar ambiguidade de colunas.

---

## 10. Exercícios

### Exercício 1 — Reconstruir `total_revenues_1997_view`

Reconstrua do zero, sem copiar o código acima. Passos:
1. Calcule a receita por item em `order_details`
2. Filtre pedidos de 1997 em `orders`
3. Faça o join e calcule a receita total do ano

Resultado esperado: um único número com a receita total de 1997.

In [0]:
# Exercício 1: Receita total de 1997
# Escreva seu código aqui

### Exercício 2 — Receita por Categoria

Calcule a receita total por categoria de produto.  
Você vai precisar fazer join de 3 tabelas:
- `order_details` (com receita calculada)
- `products` (para pegar `category_id`)
- `categories` (para pegar `category_name`)

Use `broadcast(categories)` já que é uma tabela pequena.  
Ordene por receita descendente.

In [0]:
# Exercício 2: Receita por categoria
# Escreva seu código aqui

### Exercício 3 — Clientes sem pedidos (Anti Join)

Use anti join para encontrar clientes que **nunca** fizeram pedido.  
Mostre `customer_id`, `company_name` e `country`.

**Dica:** Se o resultado for vazio, é porque todos os clientes do Northwind têm pedidos. Isso também é um resultado válido!

In [0]:
# Exercício 3: Clientes sem pedidos (anti join)
# Escreva seu código aqui